In [1]:
import os
import json
from pyspark.sql import SparkSession, functions as F

INPUT_PATH = "/user/data/preprocess/model_panel"
OUTPUT_PATH = "/user/data/preclustering"
CLUSTER_META_PATH = "/workspace/code/kshape/preclustering"
os.makedirs(CLUSTER_META_PATH, exist_ok=True)

spark = SparkSession.builder.appName("Preclustering").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

spark.conf.set("spark.sql.session.timeZone", "America/New_York")
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
spark.conf.set("spark.sql.parquet.mergeSchema", "true")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "400")

df = spark.read.parquet(INPUT_PATH)
df.printSchema()
print(f"Total input rows: {df.count()}")

df = (
  df.filter(
    F.col("pickup_bin_30m").isNotNull() & F.col("PULocationID").isNotNull()
  )
  .fillna(0, subset=["pickup_demand"])
)
df = df.filter(F.col("dataset_split") == "train")
preclustering = (
  df.select("pickup_bin_30m", "PULocationID", "pickup_demand")
    .orderBy("pickup_bin_30m", "PULocationID")
)
print(f"Preclustering rows: {preclustering.count()}")
preclustering.write.mode("overwrite").parquet(OUTPUT_PATH)

clusteringMeta = preclustering.agg(
  F.count("*").alias("rows"),
  F.countDistinct("pickup_bin_30m").alias("n_time_bins"),
  F.countDistinct("PULocationID").alias("n_locations"),
  F.min("pickup_bin_30m").alias("min_time"),
  F.max("pickup_bin_30m").alias("max_time")
).first().asDict()
with open(os.path.join(CLUSTER_META_PATH, "clustering_meta.json"), "w", encoding="utf-8") as f:
  json.dump({k: (str(v) if not isinstance(v, (int, float, str, bool, type(None))) else v) for k, v in clusteringMeta.items()}, f, indent=2, ensure_ascii=False)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6e56f68c-702b-4d73-8c0f-63b82183dc57;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 133ms :: artifacts dl 6ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-----------------------------------------------

root
 |-- pickup_bin_30m: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- time_idx: integer (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- pickup_demand: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- minute: integer (nullable = true)
 |-- slot_30m: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- is_weekday: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- demand_t_1: integer (nullable = true)
 |-- demand_t_2: integer (nullable = true)
 |-- demand_t_3: integer (nullable = true)
 |-- demand_t_4: integer (nullable = true)
 |-- demand_t_5: integer (nullable = true)
 |-- rolling_obs_24h: long (nullable = true)
 |-- rolling_max_24h: integer (nullable = true)
 |-- rolling_min_24h: integer (nullable = true)
 |-- rolling_mean_24h: double (nullable = true)
 |--

Total input rows: 13438656


Preclustering rows: 9392768
